# Transformer Block 완전 전방패스 실습 - Pre-LN Block with SwiGLU-like MLP

- Tutorial ID: `mod-transformer-block-lab`
- Tutorial: Transformer Block 완전 전방패스 실습
- Section ID: `block-1`
- Section: Pre-LN Block with SwiGLU-like MLP


In [ ]:
# ============================================================
# 코드 읽는 법 — Pre-LN Block with SwiGLU-like MLP
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.random.seed(6)
def rmsnorm(x, eps=1e-5): return x/np.sqrt(np.mean(x*x, axis=-1, keepdims=True)+eps)
def softmax(x):
        x=x-x.max(axis=-1, keepdims=True); e=np.exp(x); return e/e.sum(axis=-1, keepdims=True)
T,d,h,dh,ff = 4,8,2,4,16
X=np.random.randn(T,d)
Wq=np.random.randn(d,h*dh)/np.sqrt(d); Wk=np.random.randn(d,h*dh)/np.sqrt(d); Wv=np.random.randn(d,h*dh)/np.sqrt(d); Wo=np.random.randn(h*dh,d)/np.sqrt(h*dh)
W1=np.random.randn(d,ff)/np.sqrt(d); W2=np.random.randn(ff,d)/np.sqrt(ff)
Z=rmsnorm(X); Q=(Z@Wq).reshape(T,h,dh); K=(Z@Wk).reshape(T,h,dh); V=(Z@Wv).reshape(T,h,dh)
heads=[]; mask=np.triu(np.full((T,T),-1e9),1)
for head in range(h):
        A=softmax(Q[:,head]@K[:,head].T/np.sqrt(dh)+mask)
    heads.append(A@V[:,head])
attn_out=np.concatenate(heads,axis=-1)@Wo
X2=X+attn_out
M=np.maximum(0, rmsnorm(X2)@W1)@W2
Y=X2+M
print('residual input norm:', round(np.linalg.norm(X),3))
print('attention update norm:', round(np.linalg.norm(attn_out),3))
print('mlp update norm:', round(np.linalg.norm(M),3))
print('output shape:', Y.shape)